# Highlight Fusion — Video+Text  ⊕  Audio+Language

Late-fuse the timestamps from a **video+text** highlight detector (your frozen Qwen2.5-VL,
or a Lighthouse DETR model) with an **audio+language** detector (CLAP) — with **no training**.

**Pipeline:** each detector → per-clip saliency on a common time grid → per-video
normalize → weighted sum → peak-pick → final `(start, end, score)` highlights.

Everything loads **pretrained** models only. The notebook runs end-to-end on a
**synthetic demo** out of the box (only numpy + matplotlib needed), so you can test the
fusion plumbing first, then flip flags to plug in real inputs.

**Sections**
1. Setup & config
2. Audio utilities (extract wav from a video)
3. Detector A — Video+Text  (Qwen2.5-VL adapter  /  optional Lighthouse CG-DETR)
4. Detector B — Audio+Language (CLAP sliding-window similarity)
5. Detector C — *(optional)* Audio events via AST
6. Fusion core (score-level  +  1-D Weighted Box Fusion)
7. Run + visualize  (synthetic demo)
8. Export highlights & cut clips
9. Using it on a real video / tuning notes


## 1. Setup & config


In [1]:
# Uncomment to install (only needed for the REAL model paths, not the synthetic demo):
%uv pip install -q torch transformers librosa soundfile numpy matplotlib
# # optional: a real video+text DETR detector with pretrained checkpoints
%uv pip install -q "git+https://github.com/line/lighthouse.git"
# # ffmpeg must be on PATH for audio extraction / clip cutting:
brew install ffmpeg


SyntaxError: invalid syntax (1616130948.py, line 6)

In [1]:
import os, json, math, subprocess
import numpy as np
import matplotlib.pyplot as plt

# ---- config ----
FPS        = 2.0                          # fusion grid: 2 fps = 0.5s bins (QVHighlights convention)
WEIGHTS    = {"vt": 1.0, "audio": 0.6}    # how much to trust each detector (tune these)
CLAP_MODEL = "laion/larger_clap_general"  # or "laion/larger_clap_music" for music montages

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
print("device:", DEVICE)


device: cuda


## 2. Audio utilities


In [2]:
def extract_audio(video_path, wav_path, sr=48000):
    """Extract mono PCM wav from a video using ffmpeg (must be on PATH)."""
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-vn",
           "-ac", "1", "-ar", str(sr), "-f", "wav", str(wav_path)]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return wav_path

def load_wav(path, sr=48000):
    """Load mono audio at target sr (needs librosa)."""
    import librosa
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    return y.astype(np.float32), sr


## 3. Detector A — Video + Text

Adapter that turns your **frozen Qwen2.5-VL** output into a saliency curve on the common grid.
It matches the contract your `predict()` already emits: `{"windows": [[start,end,score],...],
"saliency": [...]}`. Prefers `windows` (with scores); falls back to the per-clip `saliency` list.


In [3]:
def vt_curve_from_qwen(pred, duration, clip_len=2.0, fps=FPS):
    """Convert a frozen-Qwen2.5-VL prediction dict to a saliency curve on the grid.

    pred = {
        "windows":  [[start_s, end_s, score], ...]   # optional; score optional
        "saliency": [s0, s1, ...]                     # optional; one score per clip_len seconds
    }
    """
    n = int(math.ceil(duration * fps))
    grid = np.zeros(n, np.float32)
    wins = pred.get("windows")
    if wins:
        for w in wins:
            s, e = float(w[0]), float(w[1])
            sc = float(w[2]) if len(w) > 2 else 1.0
            i0, i1 = int(s * fps), int(math.ceil(e * fps))
            grid[i0:min(i1, n)] = np.maximum(grid[i0:min(i1, n)], sc)
        if (grid > 0).any():
            return grid
    sal = pred.get("saliency")
    if sal is not None:
        for k, sc in enumerate(sal):
            i0 = int(math.floor(k * clip_len * fps))
            i1 = int(math.ceil((k + 1) * clip_len * fps))
            grid[i0:min(i1, n)] = float(sc)
    return grid

# ---- PLUG IN YOUR REAL MODEL HERE ----
def run_qwen_vt(video_path, query):
    """Replace this body with your actual frozen Qwen2.5-VL call.
    Must return {"windows": [[start,end,score],...]} and/or {"saliency": [...]}."""
    raise NotImplementedError("Wire up your Qwen2.5-VL predict() here.")


In [4]:
#OPTIONAL: a real pretrained video+text detector. Download the ckpt from the Lighthouse repo.
from lighthouse.models import CGDETRPredictor
_cg = CGDETRPredictor("cg_detr_qvhighlights.ckpt", device=DEVICE, feature_name="clip_slowfast")
def run_cgdetr_vt(video_path, query):
     vid = _cg.encode_video(str(video_path))
     out = _cg.predict(query, vid)
     return {"windows": out["pred_relevant_windows"], "saliency": out["pred_saliency_scores"]}


ModuleNotFoundError: No module named 'lighthouse'

## 4. Detector B — Audio + Language (CLAP)

Slides the text query over short audio windows and scores each by CLAP cosine similarity →
a language-conditioned audio-saliency curve. Fully off-the-shelf (HF `ClapModel`).


In [ ]:
_clap = {"model": None, "proc": None}

def _load_clap():
    if _clap["model"] is None:
        from transformers import ClapModel, ClapProcessor
        _clap["model"] = ClapModel.from_pretrained(CLAP_MODEL).to(DEVICE).eval()
        _clap["proc"]  = ClapProcessor.from_pretrained(CLAP_MODEL)
    return _clap["model"], _clap["proc"]

def clap_saliency(y, sr, query, win=2.0, hop=1.0, batch=64):
    """Sliding-window CLAP similarity between `query` and each audio window.
    Returns (centers_s, sims). sr should be 48000 for CLAP."""
    import torch
    model, proc = _load_clap()
    tin = proc(text=[query], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        t = torch.nn.functional.normalize(model.get_text_features(**tin), dim=-1)

    centers, segs = [], []
    for st in np.arange(0, max(0.0, len(y) / sr - win), hop):
        seg = y[int(st * sr):int((st + win) * sr)]
        if len(seg) < int(0.2 * sr):
            continue
        segs.append(seg); centers.append(st + win / 2)

    sims = []
    for i in range(0, len(segs), batch):
        ain = proc(audios=segs[i:i + batch], sampling_rate=sr,
                   return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            a = torch.nn.functional.normalize(model.get_audio_features(**ain), dim=-1)
            sims.extend((a @ t.T).squeeze(-1).cpu().numpy().tolist())
    return np.array(centers), np.array(sims, dtype=np.float32)


## 5. Detector C — Audio events via AST *(optional)*

Reuse an AudioSet sound-tagger (like the AST already in your Lore pipeline): sum the
probabilities of "exciting" classes (cheering, applause, laughter, music drops, ...) per window.


In [ ]:
EXCITING = ["cheer", "applause", "crowd", "laughter", "music", "shout", "scream", "whistl"]

def ast_excitement_curve(y, sr, win=2.0, hop=1.0):
    """Per-window sum of AudioSet 'exciting' class probabilities via AST."""
    import torch, librosa
    from transformers import AutoFeatureExtractor, ASTForAudioClassification
    name  = "MIT/ast-finetuned-audioset-10-10-0.4593"
    fe    = AutoFeatureExtractor.from_pretrained(name)
    model = ASTForAudioClassification.from_pretrained(name).to(DEVICE).eval()
    keep  = [i for i, l in model.config.id2label.items()
             if any(k in l.lower() for k in EXCITING)]
    y16 = librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y
    centers, scores = [], []
    for st in np.arange(0, max(0.0, len(y16) / 16000 - win), hop):
        seg = y16[int(st * 16000):int((st + win) * 16000)]
        inp = fe(seg, sampling_rate=16000, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            p = model(**inp).logits.softmax(-1)[0]
        centers.append(st + win / 2); scores.append(float(p[keep].sum()))
    return np.array(centers), np.array(scores, dtype=np.float32)


## 6. Fusion core


In [ ]:
def resample_to_grid(centers, vals, duration, fps=FPS):
    """Resample an irregular (centers, vals) signal onto the uniform fusion grid."""
    n = int(math.ceil(duration * fps))
    grid_t = (np.arange(n) + 0.5) / fps
    if len(centers) == 0:
        return np.zeros(n, np.float32)
    return np.interp(grid_t, centers, vals).astype(np.float32)

def znorm(x):
    sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-6 else x * 0.0

def fuse_curves(curves, weights):
    """curves: equal-length grid arrays. weights: per-detector floats. Per-video z-norm each."""
    stack = np.stack([znorm(c) for c in curves])
    w = np.asarray(weights, np.float32)[:, None]
    return (stack * w).sum(0)

def curve_to_intervals(fused, fps=FPS, percentile=70, min_len=1.0, merge_gap=0.5):
    """Threshold the fused curve at a percentile; return merged (start, end, score)."""
    thr = np.percentile(fused, percentile)
    above = fused > thr
    out, i, n = [], 0, len(fused)
    while i < n:
        if above[i]:
            j = i
            while j < n and above[j]:
                j += 1
            out.append([i / fps, j / fps, float(fused[i:j].max())]); i = j
        else:
            i += 1
    merged = []
    for iv in out:
        if merged and iv[0] - merged[-1][1] <= merge_gap:
            merged[-1][1] = iv[1]
            merged[-1][2] = max(merged[-1][2], iv[2])
        else:
            merged.append(iv)
    return [tuple(iv) for iv in merged if iv[1] - iv[0] >= min_len]


In [ ]:
# ---- Alternative: interval-level 1-D Weighted Box Fusion ----
# Use when you ONLY have discrete clips (start, end, score) and no continuous curves.
def iou_1d(a, b):
    s, e = max(a[0], b[0]), min(a[1], b[1])
    inter = max(0.0, e - s)
    union = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / union if union > 0 else 0.0

def weighted_box_fusion_1d(detections, iou_thr=0.3):
    """detections: list of (start, end, score, det_weight) from ANY number of detectors.
    Clusters overlapping intervals; returns confidence-weighted (start, end, score)."""
    clusters = []
    for s, e, sc, w in sorted(detections, key=lambda d: -d[2]):
        cw = sc * w
        for cl in clusters:
            cen = (cl["s"] / cl["wsum"], cl["e"] / cl["wsum"])
            if iou_1d((s, e), cen) >= iou_thr:
                cl["s"] += s * cw; cl["e"] += e * cw; cl["wsum"] += cw
                cl["score"] = max(cl["score"], sc); cl["n"] += 1
                break
        else:
            clusters.append({"s": s * cw, "e": e * cw, "wsum": cw, "score": sc, "n": 1})
    return [(cl["s"] / cl["wsum"], cl["e"] / cl["wsum"], cl["score"]) for cl in clusters]


## 7. Run + visualize  (synthetic demo)

Runs with **no video and no model download**: fake audio with 3 loud bursts + a fake Qwen
prediction. Set `USE_REAL_CLAP = True` (and provide real audio) to swap in the real model.


In [ ]:
def synthetic_demo(duration=60.0, sr=48000, seed=0):
    """Fake audio (3 'exciting' bursts) + matching fake Qwen windows, so the pipeline
    runs without any video or downloaded weights."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(duration * sr)) / sr
    y = 0.02 * rng.standard_normal(len(t)).astype(np.float32)        # background
    bursts = [(10, 14), (28, 33), (47, 50)]                          # loud moments
    for s, e in bursts:
        seg = slice(int(s * sr), int(e * sr))
        y[seg] += 0.3 * np.sin(2 * np.pi * 440 * t[seg]).astype(np.float32)
    # fake video+text detector: catches 2 of the 3, plus one false positive, with scores
    qwen_pred = {"windows": [[9.5, 14.5, 0.9], [27, 32, 0.8], [38, 41, 0.4]]}
    return y, sr, duration, qwen_pred, bursts

def energy_curve(y, sr, duration, win=2.0, hop=1.0):
    """RMS energy proxy — stands in for CLAP so the demo needs no model download."""
    centers, vals = [], []
    for st in np.arange(0, duration - win, hop):
        seg = y[int(st * sr):int((st + win) * sr)]
        centers.append(st + win / 2); vals.append(float(np.sqrt((seg ** 2).mean())))
    return np.array(centers), np.array(vals, np.float32)

y, sr, duration, qwen_pred, bursts = synthetic_demo()

USE_REAL_CLAP = False   # True -> use real CLAP instead of the energy proxy
if USE_REAL_CLAP:
    a_centers, a_vals = clap_saliency(y, sr, "an exciting highlight moment")
else:
    a_centers, a_vals = energy_curve(y, sr, duration)


In [ ]:
vt_curve = vt_curve_from_qwen(qwen_pred, duration, fps=FPS)
au_curve = resample_to_grid(a_centers, a_vals, duration, fps=FPS)

fused = fuse_curves([vt_curve, au_curve], [WEIGHTS["vt"], WEIGHTS["audio"]])
highlights = curve_to_intervals(fused, fps=FPS, percentile=70, min_len=1.0)

print("Fused highlights (start, end, score):")
for s, e, sc in highlights:
    print(f"  {s:6.2f} - {e:6.2f}   score={sc:.2f}")
print("ground-truth bursts:", bursts)


In [ ]:
grid_t = (np.arange(len(fused)) + 0.5) / FPS
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(grid_t, znorm(vt_curve), label="Video+Text (Qwen)", lw=2)
ax[0].plot(grid_t, znorm(au_curve), label="Audio", lw=2, alpha=0.8)
ax[0].legend(loc="upper right"); ax[0].set_title("Per-detector saliency (z-normalized)")
ax[1].plot(grid_t, fused, color="black", lw=2, label="Fused")
for s, e, sc in highlights:
    ax[1].axvspan(s, e, color="orange", alpha=0.3)
ax[1].axhline(np.percentile(fused, 70), ls="--", color="gray", lw=1, label="threshold")
ax[1].legend(loc="upper right"); ax[1].set_title("Fused curve + selected highlights")
ax[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()


## 8. Export highlights & cut clips


In [ ]:
def save_highlights(highlights, path="highlights.json"):
    data = [{"start": round(s, 3), "end": round(e, 3), "score": round(sc, 4)}
            for s, e, sc in highlights]
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print("wrote", path)
    return data

def cut_clips(video_path, highlights, outdir="clips"):
    """Cut each highlight into its own mp4 with ffmpeg (re-encodes for accurate cuts)."""
    os.makedirs(outdir, exist_ok=True)
    paths = []
    for k, (s, e, sc) in enumerate(highlights):
        out = os.path.join(outdir, f"clip_{k:02d}_{s:.1f}-{e:.1f}.mp4")
        cmd = ["ffmpeg", "-y", "-i", str(video_path), "-ss", f"{s:.2f}", "-to", f"{e:.2f}",
               "-c:v", "libx264", "-c:a", "aac", out]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        paths.append(out)
    print(f"cut {len(paths)} clips -> {outdir}/")
    return paths

save_highlights(highlights)
# cut_clips("your_video.mp4", highlights)   # uncomment with a real video


## 9. Using it on a real video / tuning notes

**Real flow**
1. `extract_audio("trip.mp4", "trip.wav")`; `y, sr = load_wav("trip.wav")`.
2. **Video+Text:** wire `run_qwen_vt()` to your frozen Qwen2.5-VL `predict()` (it already
   emits `{windows, saliency}`), or uncomment the Lighthouse CG-DETR cell.
3. **Audio+Language:** set `USE_REAL_CLAP = True`; pass your montage query
   (or use `laion/larger_clap_music` for music-driven cuts).
4. *(optional)* add `ast_excitement_curve` as a 3rd input to `fuse_curves`.
5. `fuse_curves([...], [...])` → `curve_to_intervals(...)` → `save_highlights` / `cut_clips`.

**Tuning knobs**
- `WEIGHTS` — trust per detector. Raise `audio` when the text query is weak/absent
  (montage/music) — that is exactly where audio pays off; lower it when queries are rich.
- `percentile` in `curve_to_intervals` — higher = fewer, tighter highlights.
- `min_len` / `merge_gap` — minimum highlight duration and how aggressively to stitch.

**Gotchas that decide whether it works**
- **Per-video normalization** is already applied (`znorm`) — keep it, or the larger-scaled
  detector silently dominates.
- **Temporal alignment** — every curve must share the FPS grid; pass the correct `duration`.

**Swap CLAP for a real Audio Moment Retrieval model** (Lighthouse AMR, pretrained on
Clotho-Moment / CASTELLA):
```python
# from lighthouse.models import QDDETRPredictor
# amr = QDDETRPredictor("qd_detr_clotho_moment.ckpt", device=DEVICE, feature_name="clap")
# aud = amr.encode_audio("trip.wav")
# out = amr.predict("crowd cheering", aud)
# rasterize out["pred_relevant_windows"] with vt_curve_from_qwen({"windows": out[...]}, duration)
```
